# DX Patient Features

This notebook generates patient-level features for **Dx (diagnosis) claims** using the Generic Patient Feature Engineering methodology.

## Features Generated

1. **First Occurrence** -- Days before anchor when each diagnosis first appeared in a patient's journey (full lookback)
2. **Recency** -- How recently/frequently events occurred (12M lookback, 6 sub-features)
3. **Frequency** -- Monthly claim counts, rolling lags, change in frequency, unique patient counts
4. **Hospitalization** -- Fraction of claims involving hospitalized patients (DX applicable)

## Source Mapping (DX-specific)

| Canonical | DX Raw |
|---|---|
| `EVENT_NAME` | `DIAGNOSIS_DESCRIPTION` |
| `EVENT_TYPE` | `dx` |
| `EVENT_DATE` | `SVC_DT` |

## DX-specific column naming

| Feature | Column Pattern |
|---|---|
| Freq Sub-D | `NUM_OF_PATIENTS_THE_HCP_DIAGNOSED_WITH_{EVENT}_IN_LAST_{N}_DAYS` |
| Hosp Sub-D | `NUM_OF_HOSPITALIZED_PATIENTS_THE_HCP_DIAGNOSED_IN_LAST_{N}_DAYS` |
| Hosp event collapse | `lit("ALL_DIAGNOSIS")` |

In [0]:
# ============================================================================
# CONFIG PARAMETERS (from 00_config)
# ============================================================================

# Cohort parameters
cohort_start_date = "2026-06-01"
cohort_end_date = "2026-06-01"
months_increment = 2
lookback_duration = 365
prediction_duration = 90
cohort_day_offset = 1

# Sampling parameters
sample_hcp_count = 10000                 # Randomly sample N HCPs (0 = use all HCPs)

# Feature-specific parameters
event_prevalence = 0.00
batch_size = 6

# Recency time windows (days)
time_windows_recency = [30, 60, 90, 180, 270, 360]

# Frequency lag windows (months)
lag_windows_freq = [1, 2, 3, 6, 12]

# Frequency change windows (months, excludes 12M)
change_windows_freq = [1, 2, 3, 6]

# Frequency patient count windows (days)
patient_count_windows_freq = [30, 60, 90, 180, 360]

# Hospitalization lag windows (months) -- shorter than frequency's
hosp_lag_windows = [1, 6, 12]

# Hospitalization patient count windows (days) -- NOT [30, 60, 90, 180, 360]
hosp_patient_count_windows = [30, 180, 360]

# File paths
dx1_file_path = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/dx_input_claims_all_events_features_input"
dx_hosp_file_path = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/dx_hospitalization_claims_all_events_features_input"
output_base_path = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/notebook_output/"
anchor_dates_path = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/notebook_output/hcp_model_inference_anchor_dates.csv"

print("Config parameters loaded.")

In [0]:
# ============================================================================
# IMPORTS AND STANDARDIZATION FUNCTION
# ============================================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
import pandas as pd
from datetime import datetime, timedelta

COLUMN_MAP = {
    "PROVIDER_ID": ["PROVIDER_ID", "HCP_ID", "PHYSICIAN_ID", "PROV_ID", "BHID"],
    "PATIENT_ID":  ["PATIENT_ID", "MEMBER_ID", "ENROLLEE_ID", "SUBSCRIBER_ID"],
    "EVENT_NAME":  ["PRODUCT_GROUP", "DIAGNOSIS_DESCRIPTION",
                    "PROCEDURE_DESCRIPTION", "TEST_TYPE"],
    "EVENT_DATE":  ["SVC_DT", "SERVICE_DATE", "CLAIM_DATE", "FILL_DT"],
    "CLAIM_ID":    ["CLAIM_ID", "CLAIM_NBR", "CLAIM_NUMBER"],
}

def standardize(df, event_type):
    """Standardize raw claims to canonical schema (Section 1.3)."""
    for canonical, variants in COLUMN_MAP.items():
        for v in variants:
            if v in df.columns and canonical not in df.columns:
                df = df.withColumnRenamed(v, canonical)
                break
    df = df.withColumn("EVENT_TYPE", F.lit(event_type))
    keep = [c for c in
            ["PROVIDER_ID", "PATIENT_ID", "EVENT_NAME", "EVENT_DATE",
             "EVENT_TYPE", "CLAIM_ID", "DATA_SOURCE"]
            if c in df.columns]
    return df.select(*keep)

def sanitize_column_name(name):
    """Sanitize: , removed, + -> and, / -> or, - -> _, spaces -> _, uppercase."""
    import re as _re
    name = _re.sub(r'[^A-Za-z0-9_]', '_', name.replace("+", "AND").replace("/", "_OR_"))
    name = _re.sub(r'_+', '_', name).strip('_')
    return name.upper()

print("Standardization functions loaded.")

In [0]:
# ============================================================================
# LOAD AND STANDARDIZE DX CLAIMS
# ============================================================================

# Load total DX claims (denominator for hospitalization)
dx_claims = spark.read.format("delta").load(dx1_file_path)
dx_claims = standardize(dx_claims, "dx")

# For hospitalization: also load hospitalized claims (numerator)
dx_hosp_claims = spark.read.format("delta").load(dx_hosp_file_path)
dx_hosp_claims = standardize(dx_hosp_claims, "dx")

# Ensure EVENT_DATE is date type
dx_claims = dx_claims.withColumn("EVENT_DATE", F.col("EVENT_DATE").cast("date"))
dx_hosp_claims = dx_hosp_claims.withColumn("EVENT_DATE", F.col("EVENT_DATE").cast("date"))

# Sample 10K HCPs randomly for testing (includes both old and new claims)
if sample_hcp_count > 0:
    sampled_hcps = (dx_claims
        .select("PROVIDER_ID")
        .distinct()
        .orderBy(F.rand())
        .limit(sample_hcp_count)
    )
    dx_claims = dx_claims.join(sampled_hcps, "PROVIDER_ID", "inner")
    dx_hosp_claims = dx_hosp_claims.join(sampled_hcps, "PROVIDER_ID", "inner")
    print(f"Sampled {sample_hcp_count} HCPs randomly")

print(f"DX claims loaded: {dx_claims.count()} rows")
display(dx_claims.limit(10))

In [0]:
# ============================================================================
# COHORT ANCHOR DATES GENERATION (Section 2)
# ============================================================================

def generate_cohort_anchor_dates(cohort_start, cohort_end, months_incr, lookback_dur, cohort_offset=1):
    """Generate cohort anchor dates and derived lookback boundaries."""
    anchors = []
    current = pd.Timestamp(cohort_start)
    end = pd.Timestamp(cohort_end)
    cohort_idx = 1
    while current <= end:
        cohort_label = f"I{cohort_idx:02d}"
        if cohort_offset == 1:
            lookback_end = (current - pd.DateOffset(months=2)).to_period('M').end_time.normalize()
        else:
            month_start = current.to_period('M').start_time
            if current.day >= cohort_offset:
                custom_month_start = month_start + pd.DateOffset(days=cohort_offset - 1)
            else:
                custom_month_start = (month_start - pd.DateOffset(months=1)) + pd.DateOffset(days=cohort_offset - 1)
            lookback_end = (custom_month_start - pd.DateOffset(months=1)) - pd.DateOffset(days=1)
        lookback_start = lookback_end - pd.Timedelta(days=lookback_dur)
        anchors.append({
            "COHORT": cohort_label,
            "ANCHOR_DATE": current.strftime("%Y-%m-%d"),
            "LOOKBACK_END_DATE": lookback_end.strftime("%Y-%m-%d"),
            "LOOKBACK_START_DATE": lookback_start.strftime("%Y-%m-%d"),
        })
        current = current + pd.DateOffset(months=months_incr)
        cohort_idx += 1
    return pd.DataFrame(anchors)

cohort_anchor_pd = generate_cohort_anchor_dates(
    cohort_start_date, cohort_end_date, months_increment, lookback_duration, cohort_day_offset
)
cohort_anchor_dates = spark.createDataFrame(cohort_anchor_pd)
cohort_anchor_dates = cohort_anchor_dates \
    .withColumn("ANCHOR_DATE", F.col("ANCHOR_DATE").cast("date")) \
    .withColumn("LOOKBACK_END_DATE", F.col("LOOKBACK_END_DATE").cast("date")) \
    .withColumn("LOOKBACK_START_DATE", F.col("LOOKBACK_START_DATE").cast("date"))

cohort_anchor_pd.to_csv(anchor_dates_path, index=False)

print(f"Generated {cohort_anchor_pd.shape[0]} cohorts:")
display(cohort_anchor_dates)

---
# Feature Type 1: First Occurrence (Section 3)

## What it answers
*How many days before the anchor date did this diagnosis first appear in a patient's journey?*

## How It Works
1. Collapse raw claims to (PROVIDER_ID, EVENT_NAME) with earliest EVENT_DATE
2. Cross-join with broadcast cohort anchor dates
3. Compute EVENT_TIME = datediff(LOOKBACK_END_DATE, FIRST_OCCURRENCE_DATE)
4. Filter EVENT_TIME >= 0 (future leakage guard)
5. Pivot wide by EVENT_NAME, agg max(EVENT_TIME)
6. Suffix columns with `_first_occurrence`

## Column Naming
`max(EVENT_TIME)_{EVENT_NAME}_first_occurrence`

## DX-specific: Typical event count ~17 distinct diagnoses

In [0]:
# ============================================================================
# FIRST OCCURRENCE - Step 1: Collapse raw claims (Section 3)
# ============================================================================

first_occ_intermediate = (dx_claims
    .filter(F.col("PROVIDER_ID").isNotNull())
    .filter(F.col("EVENT_NAME").isNotNull())
    .groupBy("PROVIDER_ID", "EVENT_NAME")
    .agg(F.min("EVENT_DATE").alias("FIRST_OCCURRENCE_DATE"))
)

first_occ_intermediate_path = f"{output_base_path}/dx_first_occ_intermediate"
first_occ_intermediate.write.format("parquet").mode("overwrite").save(first_occ_intermediate_path)
first_occ_intermediate = spark.read.parquet(first_occ_intermediate_path)

print(f"First occurrence intermediate: {first_occ_intermediate.count()} rows")
display(first_occ_intermediate.limit(10))

In [0]:
# ============================================================================
# FIRST OCCURRENCE - Step 2: Attach cohort dates + Step 3: Pivot wide
# ============================================================================

# Step 2: Attach cohort dates via broadcast cross-join
with_cohort = (first_occ_intermediate
    .crossJoin(F.broadcast(cohort_anchor_dates))
    .withColumn("EVENT_TIME", F.datediff(F.col("LOOKBACK_END_DATE"), F.col("FIRST_OCCURRENCE_DATE")))
    .filter(F.col("EVENT_TIME") >= 0)  # Future leakage guard (Hard)
    .withColumn("HCP_COHORT_ID", F.concat_ws("_", F.col("PROVIDER_ID"), F.col("COHORT")))
)

# Event prevalence filter (Tunable)
if event_prevalence > 0.00:
    event_counts = with_cohort.groupBy("EVENT_NAME").agg(F.countDistinct("PROVIDER_ID").alias("hcp_count"))
    total_hcps = with_cohort.select("PROVIDER_ID").distinct().count()
    events_to_keep = (event_counts
        .filter(F.col("hcp_count") / F.lit(total_hcps) >= event_prevalence)
        .select("EVENT_NAME").collect())
    keep_list = [r["EVENT_NAME"] for r in events_to_keep]
    with_cohort = with_cohort.filter(F.col("EVENT_NAME").isin(keep_list))

# Step 3: Pivot wide — batch by cohort (Section 7.5)
cohort_list = [row["COHORT"] for row in cohort_anchor_dates.select("COHORT").collect()]
batch_results = []
for i in range(0, len(cohort_list), batch_size):
    batch_cohorts = cohort_list[i:i + batch_size]
    batch_df = with_cohort.filter(F.col("COHORT").isin(batch_cohorts))
    pivoted = (batch_df
        .groupBy("HCP_COHORT_ID")
        .pivot("EVENT_NAME")
        .agg(F.max("EVENT_TIME"))
    )
    batch_results.append(pivoted)

first_occurrence_features = batch_results[0]
for batch in batch_results[1:]:
    first_occurrence_features = first_occurrence_features.unionByName(batch, allowMissingColumns=True)

# Suffix columns with _first_occurrence and sanitize names for Delta
id_cols = ["HCP_COHORT_ID"]
rename_exprs = []
for col_name in first_occurrence_features.columns:
    if col_name not in id_cols:
        sanitized = sanitize_column_name(col_name)
        rename_exprs.append(F.col(f"`{col_name}`").alias(f"MAX_EVENT_TIME_{sanitized}_FIRST_OCCURRENCE"))
    else:
        rename_exprs.append(F.col(col_name))
first_occurrence_features = first_occurrence_features.select(*rename_exprs)

first_occ_output_path = f"{output_base_path}/dx_first_occurrence"
first_occurrence_features.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(first_occ_output_path)

print(f"First occurrence features: {first_occurrence_features.count()} rows, {len(first_occurrence_features.columns)} columns")
display(first_occurrence_features.limit(10))

---
# Feature Type 2: Recency (Section 4)

## What it answers
*How recently did diagnostic events occur, and how concentrated was the activity?*

## Critical: Recency uses ONLY the 12M window (NOT full history)

## Six Sub-Features (same structure as RX)
1. Most Recent Event: `min(EVENT_TIME)_{EVENT_NAME}_time_min`
2. Total and Unique Events: `total_events`, `unique_events`
3. Counts by Time Window: `total_events_time_{N}_days`, `unique_events_time_{N}_days`
4. Counts by Type: `total_{type}`, `unique_{type}`
5. Counts by Type x Window: `total_events_time_{N}_days_{type}`, `unique_events_time_{N}_days_{type}`
6. Skewness: `skewness_time_{N}_days_{type}`

In [0]:
# ============================================================================
# RECENCY - 12M Filter + All Sub-features (Section 4)
# ============================================================================

# Cross-join claims with broadcast cohort anchor dates
recency_claims = (dx_claims
    .filter(F.col("PROVIDER_ID").isNotNull())
    .filter(F.col("EVENT_NAME").isNotNull())
    .crossJoin(F.broadcast(cohort_anchor_dates))
)

# 12M filter (Hard)
recency_claims_12m = recency_claims.filter(
    (F.col("EVENT_DATE") >= F.col("LOOKBACK_START_DATE")) &
    (F.col("EVENT_DATE") <= F.col("LOOKBACK_END_DATE"))
)

recency_claims_12m = recency_claims_12m \
    .withColumn("EVENT_TIME", F.datediff(F.col("LOOKBACK_END_DATE"), F.col("EVENT_DATE"))) \
    .withColumn("HCP_COHORT_ID", F.concat_ws("_", F.col("PROVIDER_ID"), F.col("COHORT"))) \
    .withColumn("SVC_DT_M", F.date_trunc("MM", F.col("EVENT_DATE")))

# --- Sub-feature 1: Most Recent Event (pre-aggregation pattern, Section 7.1) ---
recency_latest = (dx_claims
    .filter(F.col("PROVIDER_ID").isNotNull())
    .filter(F.col("EVENT_NAME").isNotNull())
    .groupBy("PROVIDER_ID", "EVENT_NAME")
    .agg(F.max("EVENT_DATE").alias("LATEST_EVENT_DATE"))
)
recency_sub1 = (recency_latest
    .crossJoin(F.broadcast(cohort_anchor_dates))
    .withColumn("EVENT_TIME", F.datediff(F.col("LOOKBACK_END_DATE"), F.col("LATEST_EVENT_DATE")))
    .filter(F.col("EVENT_TIME") >= 0)
    .withColumn("HCP_COHORT_ID", F.concat_ws("_", F.col("PROVIDER_ID"), F.col("COHORT")))
    .filter((F.col("LATEST_EVENT_DATE") >= F.col("LOOKBACK_START_DATE")) &
            (F.col("LATEST_EVENT_DATE") <= F.col("LOOKBACK_END_DATE")))
)
cohort_list_r = [row["COHORT"] for row in cohort_anchor_dates.select("COHORT").collect()]
batch_sub1 = []
for i in range(0, len(cohort_list_r), batch_size):
    bc = cohort_list_r[i:i + batch_size]
    bdf = recency_sub1.filter(F.col("COHORT").isin(bc))
    piv = bdf.groupBy("HCP_COHORT_ID").pivot("EVENT_NAME").agg(F.min("EVENT_TIME"))
    batch_sub1.append(piv)
recency_sub1_pivot = batch_sub1[0]
for b in batch_sub1[1:]:
    recency_sub1_pivot = recency_sub1_pivot.unionByName(b, allowMissingColumns=True)
id_cols_r = ["HCP_COHORT_ID"]
re1 = [F.col(f"`{c}`").alias(f"MIN_EVENT_TIME_{sanitize_column_name(c)}_TIME_MIN") if c not in id_cols_r else F.col(c) for c in recency_sub1_pivot.columns]
recency_sub1_result = recency_sub1_pivot.select(*re1)

# --- Sub-feature 2: Total and Unique Events ---
recency_sub2 = (recency_claims_12m
    .groupBy("HCP_COHORT_ID")
    .agg(F.count("*").alias("total_events"), F.countDistinct("EVENT_NAME").alias("unique_events"))
)

# --- Sub-feature 3: Counts by Time Window ---
recency_sub3 = recency_claims_12m.select("HCP_COHORT_ID").distinct()
for wn in time_windows_recency:
    wdf = recency_claims_12m.filter(F.col("EVENT_TIME") <= wn)
    wc = wdf.groupBy("HCP_COHORT_ID").agg(
        F.count("*").alias(f"total_events_time_{wn}_days"),
        F.countDistinct("EVENT_NAME").alias(f"unique_events_time_{wn}_days")
    )
    recency_sub3 = recency_sub3.join(wc, on="HCP_COHORT_ID", how="left")
for wn in time_windows_recency:
    recency_sub3 = recency_sub3.fillna(0, subset=[f"total_events_time_{wn}_days", f"unique_events_time_{wn}_days"])

# --- Sub-feature 4: Counts by Type ---
event_types = [r["EVENT_TYPE"] for r in recency_claims_12m.select("EVENT_TYPE").distinct().collect()]
recency_sub4 = recency_claims_12m.select("HCP_COHORT_ID").distinct()
for et in event_types:
    tc = recency_claims_12m.filter(F.col("EVENT_TYPE") == et).groupBy("HCP_COHORT_ID").agg(
        F.count("*").alias(f"total_{et}"), F.countDistinct("EVENT_NAME").alias(f"unique_{et}"))
    recency_sub4 = recency_sub4.join(tc, on="HCP_COHORT_ID", how="left")
for et in event_types:
    recency_sub4 = recency_sub4.fillna(0, subset=[f"total_{et}", f"unique_{et}"])

# --- Sub-feature 5: Counts by Type x Window ---
recency_sub5 = recency_claims_12m.select("HCP_COHORT_ID").distinct()
for et in event_types:
    for wn in time_windows_recency:
        twdf = recency_claims_12m.filter((F.col("EVENT_TYPE") == et) & (F.col("EVENT_TIME") <= wn))
        twc = twdf.groupBy("HCP_COHORT_ID").agg(
            F.count("*").alias(f"total_events_time_{wn}_days_{et}"),
            F.countDistinct("EVENT_NAME").alias(f"unique_events_time_{wn}_days_{et}"))
        recency_sub5 = recency_sub5.join(twc, on="HCP_COHORT_ID", how="left")
for et in event_types:
    for wn in time_windows_recency:
        recency_sub5 = recency_sub5.fillna(0, subset=[f"total_events_time_{wn}_days_{et}", f"unique_events_time_{wn}_days_{et}"])

# --- Sub-feature 6: Skewness ---
recency_sub6 = recency_claims_12m.select("HCP_COHORT_ID").distinct()
for et in event_types:
    for wn in time_windows_recency:
        twdf = recency_claims_12m.filter((F.col("EVENT_TYPE") == et) & (F.col("EVENT_TIME") <= wn))
        sk = twdf.groupBy("HCP_COHORT_ID").agg(F.skewness("EVENT_TIME").alias(f"skewness_time_{wn}_days_{et}"))
        recency_sub6 = recency_sub6.join(sk, on="HCP_COHORT_ID", how="left")

# Combine all sub-features
recency_combined = (recency_sub1_result
    .join(recency_sub2, on="HCP_COHORT_ID", how="left")
    .join(recency_sub3, on="HCP_COHORT_ID", how="left")
    .join(recency_sub4, on="HCP_COHORT_ID", how="left")
    .join(recency_sub5, on="HCP_COHORT_ID", how="left")
    .join(recency_sub6, on="HCP_COHORT_ID", how="left")
)

recency_combined = recency_combined.withColumn("COHORT", F.substring(F.col("HCP_COHORT_ID"), -3, 3))
recency_output_path = f"{output_base_path}/dx_recency"
recency_combined.write.format("delta").mode("overwrite").option("overwriteSchema", "true").partitionBy("COHORT").save(recency_output_path)

print(f"Recency combined: {recency_combined.count()} rows, {len(recency_combined.columns)} columns")
display(recency_combined.limit(10))

---
# Feature Type 3: Frequency (Section 5)

## What it answers
*How often does each diagnosis occur at the HCP level, and is the rate accelerating or decelerating?*

| Sub-type | What it measures | Grain |
|---|---|---|
| A | Monthly claim count per event | `[PROVIDER_ID, SVC_DT_M]` |
| B | Rolling cumulative counts (1,2,3,6,12 months) | `[PROVIDER_ID, SVC_DT_M]` |
| C | Change in frequency vs 12M baseline | `[PROVIDER_ID, SVC_DT_M]` |
| D | Unique patient counts per HCP per event per window | `[PROVIDER_ID, COHORT]` |

## DX-specific column naming for Sub-D
`NUM_OF_PATIENTS_THE_HCP_DIAGNOSED_WITH_{EVENT}_IN_LAST_{N}_DAYS`

In [0]:
# ============================================================================
# FREQUENCY - Sub-type A: Monthly Claim Count (Section 5)
# ============================================================================

freq_claims = dx_claims \
    .filter(F.col("PROVIDER_ID").isNotNull()) \
    .filter(F.col("EVENT_NAME").isNotNull()) \
    .withColumn("SVC_DT_M", F.date_trunc("MM", F.col("EVENT_DATE")))

freq_subA = (freq_claims
    .groupBy("PROVIDER_ID", "SVC_DT_M")
    .pivot("EVENT_NAME")
    .agg(F.count(F.lit(1)))
)

# Sanitize and prefix with FREQ_
id_cols_a = ["PROVIDER_ID", "SVC_DT_M"]
rename_a = [F.col(f"`{c}`").alias(f"FREQ_{sanitize_column_name(c)}") if c not in id_cols_a else F.col(c) for c in freq_subA.columns]
freq_subA = freq_subA.select(*rename_a)

freq_subA_path = f"{output_base_path}/dx_freq_subA_monthly_count"
freq_subA.write.format("delta").mode("overwrite").option("overwriteSchema", "true").partitionBy("SVC_DT_M").save(freq_subA_path)

print(f"Freq Sub-A: {freq_subA.count()} rows, {len(freq_subA.columns)} columns")
display(freq_subA.limit(10))

In [0]:
# ============================================================================
# FREQUENCY - Sub-type B: Lag Frequency (Section 5)
# ============================================================================
# Rolling cumulative counts over trailing windows [1, 2, 3, 6, 12] months
# Three-step: fill_missing -> lag -> process_and_join

def fill_missing(df, group_col="PROVIDER_ID", date_col="SVC_DT_M"):
    """Create dense spine: cross-join all HCPs with all months."""
    all_hcps = df.select(group_col).distinct()
    min_date = df.select(F.min(date_col)).collect()[0][0]
    max_date = df.select(F.max(date_col)).collect()[0][0]
    month_seq = spark.createDataFrame(
        pd.date_range(start=min_date, end=max_date, freq='MS'),
        schema=StructType([StructField(date_col, DateType(), True)])
    )
    spine = all_hcps.crossJoin(month_seq)
    result = spine.join(df, on=[group_col, date_col], how="left")
    fill_cols = [c for c in result.columns if c not in [group_col, date_col]]
    result = result.fillna(0, subset=fill_cols)
    return result

def apply_lag(df, windows, group_col="PROVIDER_ID", date_col="SVC_DT_M"):
    """Rolling window sums via Window.partitionBy().orderBy() -- batch .withColumns() to avoid deep plan."""
    w = Window.partitionBy(group_col).orderBy(date_col)
    freq_cols = [c for c in df.columns if c not in [group_col, date_col]]
    new_cols = {}
    for col_name in freq_cols:
        for n_months in windows:
            new_cols[f"{col_name}_IN_LAST_{n_months}_MONTH"] = (
                F.sum(col_name).over(w.rowsBetween(-n_months + 1, 0))
            )
    return df.withColumns(new_cols)

def process_and_join(df, windows):
    """Orchestrator: fill_missing -> lag -> drop raw FREQ columns."""
    filled = fill_missing(df)
    lagged = apply_lag(filled, windows)
    key_cols = ["PROVIDER_ID", "SVC_DT_M"]
    keep_cols = key_cols + [c for c in lagged.columns if "_IN_LAST_" in c]
    result = lagged.select(*keep_cols)
    return result

freq_subB = process_and_join(freq_subA, lag_windows_freq)

freq_subB_path = f"{output_base_path}/dx_freq_subB_lag"
freq_subB.write.format("delta").mode("overwrite").option("overwriteSchema", "true").partitionBy("SVC_DT_M").save(freq_subB_path)

print(f"Freq Sub-B: {freq_subB.count()} rows, {len(freq_subB.columns)} columns")
display(freq_subB.limit(10))

In [0]:
# ============================================================================
# FREQUENCY - Sub-type C: Change in Frequency (Section 5)
# ============================================================================
# CHANGE = (FREQ_N / N) - ((FREQ_12 - FREQ_N) / (12 - N))
# Recent windows: [1, 2, 3, 6] (12M is baseline)
# After computing, raw FREQ_*_IN_LAST_* columns are dropped.

freq_lag_cols = [c for c in freq_subB.columns if "_IN_LAST_" in c and c not in ["PROVIDER_ID", "SVC_DT_M"]]

event_names_in_freq = set()
for col_name in freq_lag_cols:
    event_base = col_name.split("_IN_LAST_")[0]
    event_names_in_freq.add(event_base)

new_cols = {}
for event_base in event_names_in_freq:
    col_12m = f"{event_base}_IN_LAST_12_MONTH"
    if col_12m not in freq_subB.columns:
        continue
    for n_months in change_windows_freq:
        col_n = f"{event_base}_IN_LAST_{n_months}_MONTH"
        if col_n not in freq_subB.columns:
            continue
        change_col = f"CHANGE_IN_FREQ_OF_HCP_CLAIMS_FOR_{event_base.replace('FREQ_', '')}_IN_LAST_{n_months}_MONTH"
        new_cols[change_col] = (
            (F.col(col_n) / F.lit(n_months)) -
            ((F.col(col_12m) - F.col(col_n)) / F.lit(12 - n_months))
        )
freq_subC = freq_subB.withColumns(new_cols)

# Select only CHANGE_IN_FREQ columns + keys
key_cols_c = ["PROVIDER_ID", "SVC_DT_M"]
change_cols = [c for c in freq_subC.columns if c.startswith("CHANGE_IN_FREQ_")]
freq_subC = freq_subC.select(*key_cols_c, *change_cols)

freq_subC_path = f"{output_base_path}/dx_freq_subC_change"
freq_subC.write.format("delta").mode("overwrite").option("overwriteSchema", "true").partitionBy("SVC_DT_M").save(freq_subC_path)

print(f"Freq Sub-C: {freq_subC.count()} rows, {len(freq_subC.columns)} columns")
display(freq_subC.limit(10))

In [0]:
# ============================================================================
# FREQUENCY - Sub-type D: Unique Patient Counts per HCP (Section 5)
# ============================================================================
# Distinct patients per HCP per diagnosis per time window
# Grain: [PROVIDER_ID, COHORT]
# DX column naming: NUM_OF_PATIENTS_THE_HCP_DIAGNOSED_WITH_{EVENT}_IN_LAST_{N}_DAYS

freq_subD_claims = (dx_claims
    .filter(F.col("PROVIDER_ID").isNotNull())
    .filter(F.col("EVENT_NAME").isNotNull())
    .filter(F.col("PATIENT_ID").isNotNull())
    .crossJoin(F.broadcast(cohort_anchor_dates))
)

freq_subD_claims = freq_subD_claims.filter(
    (F.col("EVENT_DATE") >= F.col("LOOKBACK_START_DATE")) &
    (F.col("EVENT_DATE") <= F.col("LOOKBACK_END_DATE"))
)

freq_subD_claims = freq_subD_claims.withColumn(
    "days_diff", F.datediff(F.col("LOOKBACK_END_DATE"), F.col("EVENT_DATE"))
)

# Flag per window
for wn in patient_count_windows_freq:
    freq_subD_claims = freq_subD_claims.withColumn(
        f"within_{wn}_days", F.when(F.col("days_diff") <= wn, F.lit(1)).otherwise(F.lit(0))
    )

# Dedup on [PROVIDER_ID, COHORT, EVENT_NAME, PATIENT_ID]
freq_subD_claims = freq_subD_claims.dropDuplicates(["PROVIDER_ID", "COHORT", "EVENT_NAME", "PATIENT_ID"])

freq_subD_results = []
for wn in patient_count_windows_freq:
    window_df = freq_subD_claims.filter(F.col(f"within_{wn}_days") == 1)
    counts = (window_df
        .groupBy("PROVIDER_ID", "COHORT", "EVENT_NAME")
        .agg(F.countDistinct("PATIENT_ID").alias("patient_count"))
    )
    pivoted = (counts
        .groupBy("PROVIDER_ID", "COHORT")
        .pivot("EVENT_NAME")
        .agg(F.sum("patient_count"))
    )
    # DX-specific naming
    id_cols_d = ["PROVIDER_ID", "COHORT"]
    rename_d = [F.col(f"`{c}`").alias(f"NUM_OF_PATIENTS_THE_HCP_DIAGNOSED_WITH_{sanitize_column_name(c)}_IN_LAST_{wn}_DAYS") if c not in id_cols_d else F.col(c) for c in pivoted.columns]
    pivoted = pivoted.select(*rename_d)
    freq_subD_results.append(pivoted)

freq_subD = freq_subD_results[0]
for result in freq_subD_results[1:]:
    freq_subD = freq_subD.join(result, on=["PROVIDER_ID", "COHORT"], how="left")

freq_subD_path = f"{output_base_path}/dx_freq_subD_patient_counts"
freq_subD.write.format("delta").mode("overwrite").option("overwriteSchema", "true").partitionBy("COHORT").save(freq_subD_path)

print(f"Freq Sub-D: {freq_subD.count()} rows, {len(freq_subD.columns)} columns")
display(freq_subD.limit(10))

---
# Feature Type 4: Hospitalization (Section 6)

## What it answers
*What fraction of an HCP's claims involve hospitalized patients, and how many unique hospitalized patients did the HCP see?*

## Key Design Decisions
1. **Single collapsed event** -- all claims collapsed to `lit("ALL_DIAGNOSIS")`. No per-event breakdown.
2. **Two input sources** -- total claims (denominator) + hospitalized claims `DATA_SOURCE='HX'` (numerator).
3. **HCP flags dropped** -- `KERENDIA_FLAG`, `SGLT2_GLP1_FLAG`, `NEPH_FLAG`, `TARGETING_HCP_FLAG`, `BPT_HCP_FLAG` removed.
4. **Rolling windows** -- `[1, 6, 12]` months (shorter than frequency's `[1, 2, 3, 6, 12]`).
5. **Validation filter** -- rows where `TOTAL < HOSPITALIZED` are removed (data quality).

## Sub-types

| Sub-type | What it measures | Grain | Windows |
|---|---|---|---|
| A | Total and hospitalized claim counts | `[PROVIDER_ID, SVC_DT_M]` | Monthly |
| B | Lag rolling counts (total + hosp) | `[PROVIDER_ID, SVC_DT_M]` | `[1, 6, 12]` months |
| C | Percent hospitalized | `[PROVIDER_ID, SVC_DT_M]` | `[1, 6, 12]` months |
| D | Unique hospitalized patient counts | `[PROVIDER_ID, COHORT]` | `[30, 180, 360]` days |

## Column Naming
- Sub-A: `TOTAL_CLAIMS_ALL_DIAGNOSIS`, `TOTAL_HOSPITALIZED_CLAIMS_ALL_DIAGNOSIS`
- Sub-B: `TOTAL_CLAIMS_ALL_DIAGNOSIS_IN_LAST_{N}_MONTH`, `TOTAL_HOSPITALIZED_CLAIMS_ALL_DIAGNOSIS_IN_LAST_{N}_MONTH`
- Sub-C: `PERC_HOSPITALIZED_HCP_CLAIMS_FOR_ALL_DIAGNOSIS_IN_LAST_{N}_MONTH`
- Sub-D: `NUM_OF_HOSPITALIZED_PATIENTS_THE_HCP_DIAGNOSED_IN_LAST_{N}_DAYS`

In [0]:
# ============================================================================
# HOSPITALIZATION - Sub-type A: Total and Hospitalized Claim Counts (Section 6)
# ============================================================================
# Two sources: total claims (denominator) + hospitalized claims (numerator)
# Both collapsed to single event literal: lit("ALL_DIAGNOSIS")

EVENT_COL_HOSP = F.lit("ALL_DIAGNOSIS")

# --- Total claims (denominator) ---
dx_total = (dx_claims
    .filter(F.col("PROVIDER_ID").isNotNull())
    .withColumn("SVC_DT_M", F.date_trunc("MM", F.col("EVENT_DATE")))
    .withColumn("HOSP_EVENT", EVENT_COL_HOSP)
)

total_claims_pivot = (dx_total
    .groupBy("PROVIDER_ID", "SVC_DT_M")
    .pivot("HOSP_EVENT")
    .agg(F.count(F.lit(1)))
)
# Produces: TOTAL_CLAIMS_ALL_DIAGNOSIS
total_claims_pivot = total_claims_pivot.withColumnRenamed("ALL_DIAGNOSIS", "TOTAL_CLAIMS_ALL_DIAGNOSIS")

# --- Hospitalized claims (numerator: DATA_SOURCE = 'HX') ---
dx_hosp = (dx_hosp_claims
    .filter(F.col("PROVIDER_ID").isNotNull())
    .withColumn("SVC_DT_M", F.date_trunc("MM", F.col("EVENT_DATE")))
    .withColumn("HOSP_EVENT", EVENT_COL_HOSP)
)

# Drop HCP flags (Section 6: Key Design Decision 3)
hcp_flag_cols = ["KERENDIA_FLAG", "SGLT2_GLP1_FLAG", "NEPH_FLAG", "TARGETING_HCP_FLAG", "BPT_HCP_FLAG"]
for flag in hcp_flag_cols:
    if flag in dx_hosp.columns:
        dx_hosp = dx_hosp.drop(flag)

hosp_claims_pivot = (dx_hosp
    .groupBy("PROVIDER_ID", "SVC_DT_M")
    .pivot("HOSP_EVENT")
    .agg(F.count(F.lit(1)))
)
# Produces: TOTAL_HOSPITALIZED_CLAIMS_ALL_DIAGNOSIS
hosp_claims_pivot = hosp_claims_pivot.withColumnRenamed("ALL_DIAGNOSIS", "TOTAL_HOSPITALIZED_CLAIMS_ALL_DIAGNOSIS")

# Combine: total LEFT JOIN hosp, fillna(0)
hosp_subA = total_claims_pivot.join(hosp_claims_pivot, on=["PROVIDER_ID", "SVC_DT_M"], how="left")
hosp_subA = hosp_subA.fillna(0, subset=["TOTAL_HOSPITALIZED_CLAIMS_ALL_DIAGNOSIS"])

print(f"Hosp Sub-A: {hosp_subA.count()} rows")
display(hosp_subA.limit(10))

In [0]:
# ============================================================================
# HOSPITALIZATION - Sub-type B: Lag Hospitalized Counts (Section 6)
# ============================================================================
# Rolling cumulative total and hospitalized claim counts over [1, 6, 12] months
# Same fill_missing -> lag -> process_and_join pattern as Freq Sub-B
# Validation filter: remove rows where TOTAL < HOSPITALIZED at any window

# fill_missing for hospitalization data
def fill_missing_hosp(df, group_col="PROVIDER_ID", date_col="SVC_DT_M"):
    """Create dense spine for hospitalization data."""
    all_hcps = df.select(group_col).distinct()
    min_date = df.select(F.min(date_col)).collect()[0][0]
    max_date = df.select(F.max(date_col)).collect()[0][0]
    month_seq = spark.createDataFrame(
        pd.date_range(start=min_date, end=max_date, freq='MS'),
        schema=StructType([StructField(date_col, DateType(), True)])
    )
    spine = all_hcps.crossJoin(month_seq)
    result = spine.join(df, on=[group_col, date_col], how="left")
    fill_cols = [c for c in result.columns if c not in [group_col, date_col]]
    result = result.fillna(0, subset=fill_cols)
    return result

def apply_lag_hosp(df, windows, count_cols, group_col="PROVIDER_ID", date_col="SVC_DT_M"):
    """Apply rolling window sums for specified count columns."""
    w = Window.partitionBy(group_col).orderBy(date_col)
    result = df
    for col_name in count_cols:
        for n_months in windows:
            result = result.withColumn(
                f"{col_name}_IN_LAST_{n_months}_MONTH",
                F.sum(col_name).over(w.rowsBetween(-n_months + 1, 0))
            )
    return result

# Apply fill_missing -> lag
count_cols_hosp = ["TOTAL_CLAIMS_ALL_DIAGNOSIS", "TOTAL_HOSPITALIZED_CLAIMS_ALL_DIAGNOSIS"]

hosp_filled = fill_missing_hosp(hosp_subA)
hosp_subB = apply_lag_hosp(hosp_filled, hosp_lag_windows, count_cols_hosp)

# Keep only lag columns + keys
key_cols_h = ["PROVIDER_ID", "SVC_DT_M"]
lag_cols_h = [c for c in hosp_subB.columns if "_IN_LAST_" in c]
hosp_subB = hosp_subB.select(*key_cols_h, *lag_cols_h)

# Validation filter (Hard): remove rows where TOTAL < HOSPITALIZED at any window
for n_months in hosp_lag_windows:
    total_col = f"TOTAL_CLAIMS_ALL_DIAGNOSIS_IN_LAST_{n_months}_MONTH"
    hosp_col = f"TOTAL_HOSPITALIZED_CLAIMS_ALL_DIAGNOSIS_IN_LAST_{n_months}_MONTH"
    if total_col in hosp_subB.columns and hosp_col in hosp_subB.columns:
        hosp_subB = hosp_subB.filter(F.col(total_col) >= F.col(hosp_col))

print(f"Hosp Sub-B (Lag): {hosp_subB.count()} rows, {len(hosp_subB.columns)} columns")
display(hosp_subB.limit(10))

In [0]:
# ============================================================================
# HOSPITALIZATION - Sub-type C: Percent Hospitalized (Section 6)
# ============================================================================
# PERC = WHEN TOTAL != 0 THEN HOSP / TOTAL ELSE NULL
# After computing: TOTAL_CLAIMS_*_IN_LAST_* columns are DROPPED
# Kept: PERC_HOSPITALIZED_* + TOTAL_HOSPITALIZED_* + key columns

hosp_subC = hosp_subB

for n_months in hosp_lag_windows:
    total_col = f"TOTAL_CLAIMS_ALL_DIAGNOSIS_IN_LAST_{n_months}_MONTH"
    hosp_col = f"TOTAL_HOSPITALIZED_CLAIMS_ALL_DIAGNOSIS_IN_LAST_{n_months}_MONTH"
    perc_col = f"PERC_HOSPITALIZED_HCP_CLAIMS_FOR_ALL_DIAGNOSIS_IN_LAST_{n_months}_MONTH"
    
    if total_col in hosp_subC.columns and hosp_col in hosp_subC.columns:
        hosp_subC = hosp_subC.withColumn(
            perc_col,
            F.when(F.col(total_col) != 0, F.col(hosp_col) / F.col(total_col)).otherwise(F.lit(None))
        )

# Drop TOTAL_CLAIMS_*_IN_LAST_* columns (keep PERC_HOSPITALIZED_* + TOTAL_HOSPITALIZED_* + keys)
key_cols_hc = ["PROVIDER_ID", "SVC_DT_M"]
keep_cols_hc = key_cols_hc + [c for c in hosp_subC.columns 
                              if c.startswith("PERC_HOSPITALIZED_") or c.startswith("TOTAL_HOSPITALIZED_")]
hosp_subC = hosp_subC.select(*keep_cols_hc)

print(f"Hosp Sub-C (Percent): {hosp_subC.count()} rows, {len(hosp_subC.columns)} columns")
display(hosp_subC.limit(10))

In [0]:
# ============================================================================
# HOSPITALIZATION - Sub-type D: Unique Hospitalized Patient Counts (Section 6)
# ============================================================================
# Distinct hospitalized patients per HCP within time windows from cohort's lookback end date
# Input: hospitalized claims only (DATA_SOURCE = 'HX')
# Time windows: [30, 180, 360] days (NOT [30, 60, 90, 180, 360])
# Grain: [PROVIDER_ID, COHORT]
# Column naming: NUM_OF_HOSPITALIZED_PATIENTS_THE_HCP_DIAGNOSED_IN_LAST_{N}_DAYS

hosp_subD_claims = (dx_hosp_claims
    .filter(F.col("PROVIDER_ID").isNotNull())
    .filter(F.col("PATIENT_ID").isNotNull())
    .crossJoin(F.broadcast(cohort_anchor_dates))
)

# Filter to lookback window
hosp_subD_claims = hosp_subD_claims.filter(
    (F.col("EVENT_DATE") >= F.col("LOOKBACK_START_DATE")) &
    (F.col("EVENT_DATE") <= F.col("LOOKBACK_END_DATE"))
)

# Compute days_diff
hosp_subD_claims = hosp_subD_claims.withColumn(
    "days_diff", F.datediff(F.col("LOOKBACK_END_DATE"), F.col("EVENT_DATE"))
)

# Flag per window
for wn in hosp_patient_count_windows:
    hosp_subD_claims = hosp_subD_claims.withColumn(
        f"within_{wn}_days", F.when(F.col("days_diff") <= wn, F.lit(1)).otherwise(F.lit(0))
    )

# Dedup on [PROVIDER_ID, COHORT, PATIENT_ID] (single collapsed event, no per-event dedup)
hosp_subD_claims = hosp_subD_claims.dropDuplicates(["PROVIDER_ID", "COHORT", "PATIENT_ID"])

# Count distinct patients per [PROVIDER_ID, COHORT] per window flag
hosp_subD_results = []
for wn in hosp_patient_count_windows:
    window_df = hosp_subD_claims.filter(F.col(f"within_{wn}_days") == 1)
    counts = (window_df
        .groupBy("PROVIDER_ID", "COHORT")
        .agg(F.countDistinct("PATIENT_ID").alias(f"NUM_OF_HOSPITALIZED_PATIENTS_THE_HCP_DIAGNOSED_IN_LAST_{wn}_DAYS"))
    )
    hosp_subD_results.append(counts)

# Join all window results
hosp_subD = hosp_subD_results[0]
for result in hosp_subD_results[1:]:
    hosp_subD = hosp_subD.join(result, on=["PROVIDER_ID", "COHORT"], how="left")

print(f"Hosp Sub-D (Unique Hosp Patients): {hosp_subD.count()} rows, {len(hosp_subD.columns)} columns")
display(hosp_subD.limit(10))

In [0]:
# ============================================================================
# HOSPITALIZATION - Combine Everything (Section 6)
# ============================================================================
# 1. Build skeleton_df from hcp_universe x cohort_anchor_dates with HCP_COHORT_ID
# 2. Join percent-hospitalized features on [PROVIDER_ID, SVC_DT_M]
# 3. Join num-hospitalized-patient counts on [PROVIDER_ID, COHORT]
# 4. Both join on HCP_COHORT_ID
# 5. DX: no suffix (baseline) -- only PX adds _PX suffix

# Build skeleton: hcp_universe x cohort_anchor_dates
# hcp_universe = dx_claims.select("PROVIDER_ID").distinct()  # or load from config
# skeleton_df = hcp_universe.crossJoin(cohort_anchor_dates) \
#     .withColumn("HCP_COHORT_ID", F.concat_ws("_", F.col("PROVIDER_ID"), F.col("COHORT")))

# Join percent-hospitalized on [PROVIDER_ID, SVC_DT_M] (monthly grain -> needs SVC_DT_M match)
# The hosp_subC has monthly grain; must join via SVC_DT_M then map to HCP_COHORT_ID
# skeleton_with_svc = skeleton_df.withColumn("SVC_DT_M", <derive from ANCHOR_DATE or LOOKBACK_END_DATE>)
# hosp_combined = skeleton_with_svc.join(hosp_subC, on=["PROVIDER_ID", "SVC_DT_M"], how="left")

# Join num-hospitalized-patient counts on [PROVIDER_ID, COHORT] (cohort grain)
# hosp_combined = hosp_combined.join(hosp_subD, on=["PROVIDER_ID", "COHORT"], how="left")

# Save final hospitalization output
hosp_output_path = f"{output_base_path}/dx_hospitalization"

# Save sub-outputs individually
hosp_subC_path = f"{output_base_path}/dx_hosp_subC_percent"
hosp_subC.write.format("delta").mode("overwrite").option("overwriteSchema", "true").partitionBy("SVC_DT_M").save(hosp_subC_path)

hosp_subD_path = f"{output_base_path}/dx_hosp_subD_hosp_patients"
hosp_subD.write.format("delta").mode("overwrite").option("overwriteSchema", "true").partitionBy("COHORT").save(hosp_subD_path)

print("=" * 70)
print("DX HOSPITALIZATION FEATURES - GENERATION COMPLETE")
print("=" * 70)
print()
print("Hospitalization outputs:")
print(f"  Sub-C (Percent): {hosp_subC_path}")
print(f"  Sub-D (Unique Patients): {hosp_subD_path}")
print()
print("Combine Everything (Section 6):")
print("  1. skeleton_df from hcp_universe x cohort_anchor_dates")
print("  2. LEFT JOIN hosp_subC ON [PROVIDER_ID, SVC_DT_M]")
print("  3. LEFT JOIN hosp_subD ON [PROVIDER_ID, COHORT]")
print("  4. Both join on HCP_COHORT_ID")
print("  5. DX: no suffix (baseline)")
print("=" * 70)

In [0]:
# ============================================================================
# DX PATIENT FEATURES - FINAL OUTPUT SUMMARY (Section 8)
# ============================================================================

print("=" * 70)
print("DX PATIENT FEATURES - GENERATION COMPLETE")
print("=" * 70)
print()
print("Feature outputs generated:")
print()
print("1. First Occurrence (HCP_COHORT_ID grain)")
print(f"   Path: {output_base_path}/dx_first_occurrence")
print()
print("2. Recency (HCP_COHORT_ID grain, 6 sub-features)")
print(f"   Path: {output_base_path}/dx_recency")
print()
print("3. Frequency:")
print(f"   Sub-A (Monthly): {output_base_path}/dx_freq_subA_monthly_count")
print(f"   Sub-B (Lag): {output_base_path}/dx_freq_subB_lag")
print(f"   Sub-C (Change): {output_base_path}/dx_freq_subC_change")
print(f"   Sub-D (Patients): {output_base_path}/dx_freq_subD_patient_counts")
print()
print("4. Hospitalization:")
print(f"   Sub-C (Percent): {output_base_path}/dx_hosp_subC_percent")
print(f"   Sub-D (Hosp Patients): {output_base_path}/dx_hosp_subD_hosp_patients")
print()
print("=" * 70)
print("Join strategy (Section 8.2):")
print("  Step 1: Build skeleton from hcp_universe x cohort_anchor_dates")
print("  Step 2: LEFT JOIN first_occurrence ON HCP_COHORT_ID")
print("  Step 3: LEFT JOIN recency ON HCP_COHORT_ID")
print("  Step 4: LEFT JOIN freq_subABC ON [PROVIDER_ID, SVC_DT_M]")
print("  Step 5: LEFT JOIN freq_subD ON [PROVIDER_ID, COHORT]")
print("  Step 6: LEFT JOIN hosp_subC ON [PROVIDER_ID, SVC_DT_M]")
print("  Step 7: LEFT JOIN hosp_subD ON [PROVIDER_ID, COHORT]")
print("  Step 8: DX features: no suffix (baseline)")
print("=" * 70)